# Cross-Staff Calibration Campaign 001

**Notebook:** 02 Descriptive Statistics  
**Survey:** SUR-CAL-2026-001  
**Purpose:** Quantify the central tendency, spread, and repeatability of the calibration observations.  
**Author:** Dennis Hazelett

## 1. Introduction

This notebook follows the exploratory analysis and provides descriptive summaries of the first Kepler cross-staff calibration survey.

The emphasis remains descriptive. No calibration model is fitted here.

The canonical survey package remains unchanged:

`data/examples/SUR-CAL-2026-001/`

## 2. Import Packages

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.figsize'] = (8, 5)

## 3. Load the Canonical Survey

In [ ]:
# Find the repository root by walking upward until data/ is found.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'data').exists():
    repo_root = repo_root.parent

survey_dir = repo_root / 'data' / 'examples' / 'SUR-CAL-2026-001'

with open(survey_dir / 'survey.json', encoding='utf-8') as f:
    survey = json.load(f)

observations = pd.read_csv(survey_dir / 'observations.csv')
notes = pd.read_csv(survey_dir / 'notes.csv')

print(f"Loaded {len(observations)} observations from {survey['survey_id']}")

## 4. Prepare Analysis-Friendly Columns

The canonical column names are preserved. Short aliases are created only inside this notebook to make the analysis easier to read.

In [ ]:
cal = observations.copy()

# Support the canonical dotted calibration field names.
column_aliases = {
    'calibration.target_distance': 'target_distance',
    'calibration.target_width': 'target_width',
    'calibration.target_id': 'target_id',
}

for source, alias in column_aliases.items():
    if source in cal.columns:
        cal[alias] = cal[source]

required = ['staff_reading', 'fiducial_id', 'target_distance', 'target_width']
missing = [name for name in required if name not in cal.columns]
if missing:
    raise KeyError(f'Missing required analysis columns: {missing}')

cal.head()

## 5. Overall Descriptive Statistics

In [ ]:
overall_summary = cal[
    ['staff_reading', 'target_distance', 'target_width']
].describe().T

overall_summary

Record initial observations here:

- Which variables have the widest range?
- Are the means and medians similar?
- Does any variable appear strongly skewed?

## 6. Counts by Fiducial

In [ ]:
fiducial_counts = (
    cal.groupby('fiducial_id')
    .size()
    .rename('observation_count')
    .sort_values(ascending=False)
)

fiducial_counts

## 7. Staff Reading Summary by Fiducial

In [ ]:
fiducial_summary = (
    cal.groupby('fiducial_id')['staff_reading']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .sort_index()
)

fiducial_summary

The standard deviation summarizes spread within each fiducial group. Groups with very few observations should be interpreted cautiously.

## 8. Repeated-Measurement Groups

In [ ]:
group_columns = ['fiducial_id', 'target_distance', 'target_width']

repeatability_summary = (
    cal.groupby(group_columns)['staff_reading']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .reset_index()
    .sort_values(group_columns)
)

repeatability_summary

Groups with `count > 1` represent repeated measurements under the same recorded calibration conditions.

In [ ]:
repeated_groups = repeatability_summary[
    repeatability_summary['count'] > 1
].copy()

repeated_groups

## 9. Within-Group Spread

In [ ]:
repeated_groups['range'] = repeated_groups['max'] - repeated_groups['min']

repeated_groups[
    ['fiducial_id', 'target_distance', 'target_width', 'count', 'std', 'range']
].sort_values('range', ascending=False)

This table highlights repeated-measurement groups with the largest observed spread. It is descriptive only and does not establish whether any observation is erroneous.

## 10. Boxplot of Staff Readings by Fiducial

In [ ]:
fiducial_order = sorted(cal['fiducial_id'].dropna().unique())
plot_data = [
    cal.loc[cal['fiducial_id'] == fiducial, 'staff_reading']
    for fiducial in fiducial_order
]

plt.boxplot(plot_data, tick_labels=fiducial_order)
plt.title('Staff Readings by Fiducial')
plt.xlabel('Fiducial')
plt.ylabel('Staff Reading')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Group Means Across Target Distance

In [ ]:
group_means = (
    cal.groupby(group_columns, as_index=False)['staff_reading']
    .mean()
    .rename(columns={'staff_reading': 'mean_staff_reading'})
)

for fiducial, subset in group_means.groupby('fiducial_id'):
    plt.scatter(
        subset['target_distance'],
        subset['mean_staff_reading'],
        label=fiducial,
    )

plt.title('Mean Staff Reading vs Target Distance')
plt.xlabel('Target Distance')
plt.ylabel('Mean Staff Reading')
plt.legend(title='Fiducial')
plt.tight_layout()
plt.show()

## 12. Observations Linked to Notes

In [ ]:
if {'notes_id', 'observation_id', 'note'}.issubset(notes.columns):
    noted_observations = cal.merge(
        notes,
        on='observation_id',
        how='inner',
        suffixes=('', '_note'),
    )
else:
    noted_observations = notes.copy()

noted_observations

Review these observations before interpreting unusually large or small values.

## 13. Save Derived Summary Tables

In [ ]:
output_dir = (
    repo_root / 'analysis' / 'tables' / 'cross-staff-calibration-001'
)
output_dir.mkdir(parents=True, exist_ok=True)

overall_summary.to_csv(output_dir / 'overall-summary.csv')
fiducial_summary.to_csv(output_dir / 'fiducial-summary.csv')
repeatability_summary.to_csv(output_dir / 'repeatability-summary.csv', index=False)

print(f'Saved derived tables to: {output_dir}')

## 14. Interpretation Notes

Use this section to record what the descriptive statistics show.

Suggested prompts:

- Which repeated-measurement groups are most consistent?
- Does spread appear to change with target distance?
- Are some fiducials associated with greater variability?
- Do the observer notes correspond to unusual measurements?
- Which patterns should be investigated in the calibration-model notebook?

## 15. Next Step

The next notebook may begin calibration-model development, but only after the descriptive results have been reviewed and the scientific questions have been refined.

## 16. Anomalous observations

For the 1/2" calibration target there is a wide variance. Is it caused by a single outlier or inherent noise to this measurement cluster?

The largest within-group spread occurred for the 0.5-inch target observed with the 0.25-inch fiducial. This configuration approached the practical resolution limit of the instrument, requiring an additional measurement distance (48 inches) because the standard 120-inch distance was not measurable. At present, this observation is retained as a valid measurement representing the operational limits of the instrument rather than treated as an erroneous outlier. Future calibration campaigns may revisit this configuration with additional repeated measurements.

In [ ]:
subset = cal[
    (cal["fiducial_id"] == 0.25) &
    (cal["target_width"] == 0.5)
].sort_values("target_distance")

subset[
    [
        "observation_id",
        "target_distance",
        "staff_reading",
        "notes_id",
    ]
]

plt.figure(figsize=(10,4))

plt.plot(
    range(len(subset)),
    subset["staff_reading"],
    "o-",
)

plt.xticks(range(len(subset)), subset["observation_id"], rotation=90)

plt.xlabel("Observation")
plt.ylabel("Staff Reading")

plt.title("Measurement Sequence")

plt.tight_layout()
plt.show()

One more plot showing the 1/2" target observatins relative to all others.

In [ ]:
plt.figure(figsize=(8, 6))

# All observations
plt.scatter(
    cal["target_distance"],
    cal["staff_reading"],
    color="0.6",          # medium gray
    alpha=0.8,
    label="All observations",
)

# Highlight the 0.5" targets
mask = cal["target_width"] == 0.5

plt.scatter(
    cal.loc[mask, "target_distance"],
    cal.loc[mask, "staff_reading"],
    color="purple",
    s=70,
    label='0.5" target',
)

plt.xlabel("Target Distance (in)")
plt.ylabel("Staff Reading (in)")
plt.title("Cross-Staff Calibration Measurements")

plt.legend()
plt.grid(True)

plt.show()